# Cardiac Ultrasound — Full Inference Pipeline

This notebook runs the **trained models** on new ultrasound images.

**You do NOT need the training dataset (database_nifti/) to use this notebook.**

### What you need (get all `.pt` files from your teammate):
| Weight File | Purpose |
|---|---|
| `segmentation_weights.pt` | U-Net segmentation (LV, myocardium, LA) |
| `cactus_resnet18_classification.pt` | Classify ultrasound view type (2CH, 4CH, etc.) |
| `cactus_resnet18_regression.pt` | Predict image quality score for each angle |
| `ef_trained.pt` | Predict Ejection Fraction directly |
| `multitask_ultrasound_model.pt` | Combined multi-task model |

- Your ultrasound images (`.nii.gz`, `.png`, `.jpg`, or `.dcm`)

### What this does:
1. **Classifies** what ultrasound view/angle the image shows
2. **Assesses quality** — is this a good angle?
3. **Segments** cardiac structures (LV cavity, myocardium, left atrium)
4. **Calculates metrics** (areas, volumes, EF, VTI, cardiac output)
5. **Visualizes** results with color overlays

In [6]:
# Cell [1] — Create a local virtual environment and install packages
import subprocess, sys, os

venv_path = os.path.expanduser("~/cardiac_venv")

# Create venv (only needs to run once)
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
    print(f"✅ Created venv at {venv_path}")

# Install packages into the venv
pip_path = os.path.join(venv_path, "bin", "pip")
subprocess.run([pip_path, "install", "torch", "torchvision", "numpy", "nibabel", "matplotlib", "Pillow", "opencv-python-headless", "timm"], check=True)

# Clone PanEcho repo (needed for the ConvNeXt video EF model)
panecho_dir = os.path.expanduser("~/PanEcho")
if not os.path.exists(panecho_dir):
    subprocess.run(["git", "clone", "https://github.com/cards-yale/panecho.git", panecho_dir], check=True)
    print(f"Cloned PanEcho to {panecho_dir}")
else:
    print(f"PanEcho already at {panecho_dir}")

# Add venv packages to this notebook's path
import site
site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
site.addsitedir(site_packages)

print(f"\n✅ All packages installed and available!")
print(f"   Location: {site_packages}")

✅ Created venv at /home/users/leah26/cardiac_venv
  Using cached nibabel-5.3.3-py3-none-any.whl.metadata (9.1 kB)
  Using cached filelock-3.24.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.m


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/users/leah26/cardiac_venv/bin/python3.13 -m pip install --upgrade pip



✅ All packages installed and available!
   Location: /home/users/leah26/cardiac_venv/lib/python3.13/site-packages


In [7]:
# Cell [2] — Imports
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.transforms import functional as F
import torchvision.transforms as T
from pathlib import Path
import cv2

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Add PanEcho to path for FrameTransformer
panecho_dir = os.path.expanduser("~/PanEcho")
for p in [panecho_dir, os.path.join(panecho_dir, "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from src.models import FrameTransformer
    PANECHO_AVAILABLE = True
    print(f"PanEcho FrameTransformer imported from {panecho_dir}")
except ImportError as e:
    PANECHO_AVAILABLE = False
    print(f"PanEcho not available ({e}) — ConvNeXt EF models will be skipped")

Using device: cpu


In [ ]:
# Cell [3] — Configuration
# =====================================================
# UPDATE THESE PATHS to where YOU saved each .pt file
# =====================================================

WEIGHTS_DIR = "weights/"  # Folder containing all .pt files — CHANGE THIS

# --- Segmentation ---
SEG_MODEL_PATH    = os.path.join(WEIGHTS_DIR, "segmentation_weights.pt")

# --- Classification (best version) ---
CLASS_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "best_cactus_model.pt")

# --- Quality (standalone) ---
QUALITY_MODEL_PATH = os.path.join(WEIGHTS_DIR, "quality_model.pt")

# --- EF models ---
# ConvNeXt video models (16 frames, best accuracy) — tried first
EF_CONVNEXT_CANDIDATES = [
    os.path.join(WEIGHTS_DIR, "ef_best.pt"),
    os.path.join(WEIGHTS_DIR, "ef_finetuned.pt"),
    os.path.join(WEIGHTS_DIR, "ef_model_final.pt"),
    os.path.join(WEIGHTS_DIR, "ef_frozen_head.pt"),
    os.path.join(WEIGHTS_DIR, "ef_model.pt"),
]
# ResNet18 multi-head fallbacks (single frame)
EF_RESNET_CANDIDATES = [
    os.path.join(WEIGHTS_DIR, "ef_trained.pt"),
    os.path.join(WEIGHTS_DIR, "multitask_ultrasound_model.pt"),
]

# --- Multitask ---
MULTI_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "multitask_ultrasound_model.pt")

IMAGE_DIR = "test_images/"  # Folder with your ultrasound images — CHANGE THIS

IMG_SIZE = (256, 256)

LABEL_MAP = {
    0: 'Background',
    1: 'LV Cavity',
    2: 'Myocardium',
    3: 'Left Atrium'
}

print("Config loaded")
print(f"  Weights dir:  {WEIGHTS_DIR}")
print(f"  Image dir:    {IMAGE_DIR}")
print(f"  Input size:   {IMG_SIZE}")
print()
print("  Weight files:")
for name, path in [("Segmentation", SEG_MODEL_PATH),
                    ("Classification", CLASS_MODEL_PATH),
                    ("Quality", QUALITY_MODEL_PATH),
                    ("Multitask", MULTI_MODEL_PATH)]:
    status = "FOUND" if os.path.exists(path) else "NOT FOUND"
    print(f"    {name:<16} {status:<12} {Path(path).name}")
print(f"  EF ConvNeXt candidates (video-based, best accuracy):")
for p in EF_CONVNEXT_CANDIDATES:
    status = "FOUND" if os.path.exists(p) else "NOT FOUND"
    print(f"    {status:<12} {Path(p).name}")
print(f"  EF ResNet18 fallbacks (single-frame):")
for p in EF_RESNET_CANDIDATES:
    status = "FOUND" if os.path.exists(p) else "NOT FOUND"
    print(f"    {status:<12} {Path(p).name}")

In [ ]:
# Cell [3b] — Inspect weight files to discover architectures

def inspect_weights(path, label=""):
    """Print summary of a .pt weight file's contents."""
    if not os.path.exists(path):
        print(f"  [{label}] File not found: {path}\n")
        return None

    try:
        data = torch.load(path, map_location="cpu", weights_only=False)
    except (RuntimeError, EOFError, Exception) as e:
        print(f"  [{label}] {path}")
        print(f"    FAILED to load: {type(e).__name__}: {e}")
        print(f"    (File may be corrupted or incomplete.)\n")
        return None

    print(f"  [{label}] {path}")
    print(f"    Top-level type: {type(data).__name__}")

    state_dict = None
    if isinstance(data, dict):
        top_keys = list(data.keys())[:20]
        print(f"    Keys: {top_keys}")
        if "state_dict" in data:
            state_dict = data["state_dict"]
        elif "model_state_dict" in data:
            state_dict = data["model_state_dict"]
        elif all(isinstance(v, torch.Tensor) for v in list(data.values())[:5]):
            state_dict = data
    elif isinstance(data, nn.Module):
        print(f"    Saved as full model: {type(data).__name__}")
        state_dict = data.state_dict()

    if state_dict is not None:
        keys = list(state_dict.keys())
        print(f"    Total layers: {len(keys)}")
        print(f"    All Conv2d weight shapes:")
        for k in keys:
            if "weight" in k and state_dict[k].ndim == 4:
                print(f"      {k}: {list(state_dict[k].shape)}")
        print(f"    Final layer:")
        for k in keys[-3:]:
            print(f"      {k}: {list(state_dict[k].shape) if state_dict[k].ndim > 0 else 'scalar'}")
    print()
    return data

print("=== Weight File Inspection ===\n")
for label, path in [("Segmentation", SEG_MODEL_PATH),
                     ("Classification", CLASS_MODEL_PATH),
                     ("Quality", QUALITY_MODEL_PATH),
                     ("Multitask", MULTI_MODEL_PATH)]:
    inspect_weights(path, label)

print("--- EF ConvNeXt candidates (video-based) ---\n")
for ef_path in EF_CONVNEXT_CANDIDATES:
    inspect_weights(ef_path, f"EF:{Path(ef_path).name}")

print("--- EF ResNet18 fallbacks (single-frame) ---\n")
for ef_path in EF_RESNET_CANDIDATES:
    inspect_weights(ef_path, f"EF:{Path(ef_path).name}")

In [ ]:
# Cell [4] — Model Architectures
# Matched to the ACTUAL weight files discovered by Cell [3b].

import torchvision.models as models

# ---------------------------------------------------------------------------
# Segmentation: SimpleUNet  (matches segmentation_weights.pt)
#   Keys: down1, down2, mid, up2, up1, out
#   No BatchNorm, 2-level encoder, bilinear upsampling, 1-channel output
# ---------------------------------------------------------------------------

class SimpleUNet(nn.Module):
    """
    Lightweight U-Net that matches segmentation_weights.pt.
    Auto-detects channel sizes from a state dict if provided.
    """
    def __init__(self, in_ch=1, out_ch=1, features=(64, 128, 256)):
        super().__init__()
        f1, f2, f3 = features
        self.down1 = self._block(in_ch, f1)
        self.down2 = self._block(f1, f2)
        self.mid   = self._block(f2, f3)
        self.up2   = self._block(f3 + f2, f2)
        self.up1   = self._block(f2 + f1, f1)
        self.out   = nn.Conv2d(f1, out_ch, 1)
        self.pool  = nn.MaxPool2d(2)

    @staticmethod
    def _block(in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        m  = self.mid(self.pool(d2))
        u2 = self.up2(torch.cat([
            nn.functional.interpolate(m, size=d2.shape[2:], mode='bilinear', align_corners=True),
            d2], dim=1))
        u1 = self.up1(torch.cat([
            nn.functional.interpolate(u2, size=d1.shape[2:], mode='bilinear', align_corners=True),
            d1], dim=1))
        return self.out(u1)

    @classmethod
    def from_state_dict(cls, sd):
        """Build a SimpleUNet whose channels match the given state dict."""
        in_ch = sd['down1.0.weight'].shape[1]
        f1    = sd['down1.0.weight'].shape[0]
        f2    = sd['down2.0.weight'].shape[0]
        f3    = sd['mid.0.weight'].shape[0]
        out_ch = sd['out.weight'].shape[0]
        model = cls(in_ch, out_ch, (f1, f2, f3))
        model.load_state_dict(sd)
        return model


# ---------------------------------------------------------------------------
# Classification / Regression: ResNet18-based
# ---------------------------------------------------------------------------

def _extract_state_dict(raw):
    """Pull the actual state dict out of whatever torch.load returned."""
    if isinstance(raw, dict):
        if "state_dict" in raw:
            return raw["state_dict"]
        if "model_state_dict" in raw:
            return raw["model_state_dict"]
        if all(isinstance(v, torch.Tensor) for v in list(raw.values())[:3]):
            return raw
    return None


def load_resnet18_from_weights(path, fallback_outputs=1):
    """
    Load a ResNet18 from a .pt file, auto-detecting:
      - number of input channels (1 for grayscale, 3 for RGB)
      - number of output classes/values
    """
    if not os.path.exists(path):
        print(f"  File not found: {path}")
        return None

    try:
        raw = torch.load(path, map_location=DEVICE, weights_only=False)
    except RuntimeError as e:
        print(f"  Failed to load {path}: {e}")
        return None

    sd = _extract_state_dict(raw)
    if sd is None:
        print(f"  Unexpected format in {path} (type={type(raw).__name__})")
        return None

    # Auto-detect shapes
    in_ch = sd['conv1.weight'].shape[1] if 'conv1.weight' in sd else 1
    n_out = sd['fc.weight'].shape[0] if 'fc.weight' in sd else fallback_outputs

    net = models.resnet18(weights=None)
    net.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
    net.fc = nn.Linear(net.fc.in_features, n_out)
    net.load_state_dict(sd, strict=False)
    net = net.to(DEVICE)
    net.eval()
    return net, n_out, in_ch


# ---------------------------------------------------------------------------
# Multi-Head ResNet18: matches ef_trained.pt
#   Keys: backbone.* (ResNet18), quality_head.*, ef_head.*, lvot_head.*
#   Three separate prediction heads sharing one backbone.
# ---------------------------------------------------------------------------

class MultiHeadResNet(nn.Module):
    """
    ResNet18 backbone with auto-detected prediction heads.
    Handles any combination of *_head layers (e.g., quality_head, ef_head,
    grade_head, lvot_head) found in the state dict.
    """
    def __init__(self, in_channels=1, heads=None):
        """
        Args:
            in_channels: 1 for grayscale, 3 for RGB
            heads: dict of {head_name: num_outputs}, e.g. {'ef': 1, 'quality': 3}
        """
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(
            in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_features = self.backbone.fc.in_features  # 512
        self.backbone.fc = nn.Identity()

        heads = heads or {}
        self.head_names = sorted(heads.keys())
        for name, n_out in heads.items():
            setattr(self, f"{name}_head", nn.Linear(num_features, n_out))

    def forward(self, x):
        features = self.backbone(x)
        return {name: getattr(self, f"{name}_head")(features)
                for name in self.head_names}

    @classmethod
    def from_state_dict(cls, sd, device='cpu'):
        in_ch = sd['backbone.conv1.weight'].shape[1]
        heads = {}
        for k in sd:
            if k.endswith('_head.weight') and sd[k].ndim == 2:
                name = k.replace('_head.weight', '')
                heads[name] = sd[k].shape[0]
        model = cls(in_channels=in_ch, heads=heads)
        model.load_state_dict(sd)
        return model.to(device).eval()


# ---------------------------------------------------------------------------
# ConvNeXt Video EF Model (matches ef_model_final.pt, ef_finetuned.pt, etc.)
#   Architecture: nn.Sequential(FrameTransformer(convnext_tiny), nn.Linear(768,1))
#   Input: (B, 3, 16, 224, 224) — 16 RGB frames, ImageNet-normalized
# ---------------------------------------------------------------------------

# Video frame preprocessing (same as teammate's training code)
VIDEO_EF_TRANSFORM = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

N_FRAMES = 16

def read_video_for_ef(path, n_frames=N_FRAMES):
    """
    Read n_frames evenly-spaced frames from a video, preprocessed for EF model.
    Returns tensor of shape (3, T, 224, 224).
    """
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = set(np.linspace(0, max(length - 1, 0), n_frames).astype(int))

    frames = []
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in idxs:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(VIDEO_EF_TRANSFORM(frame_rgb))
        i += 1
    cap.release()

    while len(frames) < n_frames:
        frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))

    return torch.stack(frames, dim=1)  # (3, T, 224, 224)


def build_convnext_ef_model():
    """Build the ConvNeXt video EF model (requires PanEcho)."""
    if not PANECHO_AVAILABLE:
        return None
    backbone = FrameTransformer(
        arch="convnext_tiny",
        n_heads=8,
        n_layers=2,
        transformer_dropout=0.1,
        pooling="mean",
        clip_len=N_FRAMES)
    return nn.Sequential(backbone, nn.Linear(768, 1))


print(f"SimpleUNet defined     (matches segmentation_weights.pt)")
print(f"ResNet18 loader defined (matches cactus_resnet18_*.pt)")
print(f"MultiHeadResNet defined (matches ef_trained.pt, multitask_ultrasound_model.pt)")
if PANECHO_AVAILABLE:
    print(f"ConvNeXt EF builder defined (matches ef_model_final.pt, ef_finetuned.pt, etc.)")
else:
    print(f"ConvNeXt EF builder UNAVAILABLE (PanEcho not installed)")

In [ ]:
# Cell [5] — Load All Trained Models

loaded_models = {}
model_info = {}


def _try_load_state_dict(path):
    """Load a .pt file and extract the state dict, handling various formats."""
    try:
        raw = torch.load(path, map_location=DEVICE, weights_only=False)
    except (RuntimeError, EOFError, Exception) as e:
        print(f"    Failed to read {Path(path).name}: {type(e).__name__}: {e}")
        return None
    sd = _extract_state_dict(raw)
    if sd is None and isinstance(raw, dict):
        sd = raw
    return sd


def _has_backbone_prefix(sd):
    """Check if keys use 'backbone.' prefix (MultiHeadResNet format)."""
    return any(k.startswith('backbone.') for k in sd)


# ──────────────────────────────────────────────────────────────────────
# 1. Segmentation (SimpleUNet — binary output, raw 0-255 input)
# ──────────────────────────────────────────────────────────────────────
if os.path.exists(SEG_MODEL_PATH):
    try:
        sd = _try_load_state_dict(SEG_MODEL_PATH)
        seg_model = SimpleUNet.from_state_dict(sd).to(DEVICE)
        seg_model.eval()
        loaded_models["segmentation"] = seg_model
        model_info["segmentation"] = {"out_channels": sd['out.weight'].shape[0]}
        print(f"Segmentation loaded from {Path(SEG_MODEL_PATH).name}")
    except Exception as e:
        print(f"Segmentation failed: {e}")
else:
    print(f"Segmentation not found: {Path(SEG_MODEL_PATH).name}")

# ──────────────────────────────────────────────────────────────────────
# 2. Classification / Grade / Quality (may be multi-head with backbone.*)
# ──────────────────────────────────────────────────────────────────────
if os.path.exists(CLASS_MODEL_PATH):
    sd = _try_load_state_dict(CLASS_MODEL_PATH)
    if sd is not None and _has_backbone_prefix(sd):
        try:
            cls_mh = MultiHeadResNet.from_state_dict(sd, device=DEVICE)
            loaded_models["class_multihead"] = cls_mh
            in_ch = sd['backbone.conv1.weight'].shape[1]
            heads = {k.replace('_head.weight', ''): sd[k].shape[0]
                     for k in sd if k.endswith('_head.weight') and sd[k].ndim == 2}
            model_info["class_multihead"] = {"in_channels": in_ch, "heads": heads}
            print(f"Classification loaded (multi-head: {heads}, {in_ch}-ch) from {Path(CLASS_MODEL_PATH).name}")
        except Exception as e:
            print(f"Classification multi-head failed: {e}")
    elif sd is not None:
        result = load_resnet18_from_weights(CLASS_MODEL_PATH)
        if result:
            loaded_models["classification"] = result[0]
            model_info["classification"] = {"n_classes": result[1], "in_channels": result[2]}
            print(f"Classification loaded (ResNet18, {result[1]} classes) from {Path(CLASS_MODEL_PATH).name}")
else:
    print(f"Classification not found: {Path(CLASS_MODEL_PATH).name}")

# ──────────────────────────────────────────────────────────────────────
# 3. Quality model (standalone)
# ──────────────────────────────────────────────────────────────────────
if os.path.exists(QUALITY_MODEL_PATH):
    sd = _try_load_state_dict(QUALITY_MODEL_PATH)
    if sd is not None:
        if _has_backbone_prefix(sd):
            try:
                qm = MultiHeadResNet.from_state_dict(sd, device=DEVICE)
                loaded_models["quality_multihead"] = qm
                model_info["quality_multihead"] = {"in_channels": sd['backbone.conv1.weight'].shape[1]}
                print(f"Quality loaded (multi-head) from {Path(QUALITY_MODEL_PATH).name}")
            except Exception as e:
                print(f"Quality (multi-head) failed: {e}")
        else:
            result = load_resnet18_from_weights(QUALITY_MODEL_PATH)
            if result:
                loaded_models["quality_standalone"] = result[0]
                model_info["quality_standalone"] = {"n_outputs": result[1], "in_channels": result[2]}
                print(f"Quality loaded (ResNet18, {result[1]} outputs) from {Path(QUALITY_MODEL_PATH).name}")
else:
    print(f"Quality not found: {Path(QUALITY_MODEL_PATH).name}")

# ──────────────────────────────────────────────────────────────────────
# 4. EF model — try ConvNeXt video models first (best accuracy),
#    then fall back to ResNet18 multi-head (single frame)
# ──────────────────────────────────────────────────────────────────────
ef_loaded = False

# 4a. Try ConvNeXt video models (16 frames, requires PanEcho)
if PANECHO_AVAILABLE:
    for ef_path in EF_CONVNEXT_CANDIDATES:
        if not os.path.exists(ef_path):
            continue
        sd = _try_load_state_dict(ef_path)
        if sd is None:
            continue
        is_convnext = any('encoder.model.features' in k for k in list(sd.keys())[:30])
        if not is_convnext:
            continue
        try:
            convnext_model = build_convnext_ef_model()
            has_backbone_wrap = any(k.startswith('backbone.') for k in sd)
            if has_backbone_wrap:
                stripped = {k.replace('backbone.', '0.').replace('head.', '1.'):v for k,v in sd.items()}
                convnext_model.load_state_dict(stripped)
            else:
                convnext_model.load_state_dict(sd)
            convnext_model = convnext_model.to(DEVICE).eval()
            loaded_models["ef_convnext"] = convnext_model
            model_info["ef_convnext"] = {"type": "ConvNeXt+Transformer", "n_frames": N_FRAMES}
            print(f"EF loaded (ConvNeXt video, {N_FRAMES} frames) from {Path(ef_path).name}")
            ef_loaded = True
            break
        except Exception as e:
            print(f"  {Path(ef_path).name} ConvNeXt failed: {e}")

# 4b. Try ResNet18 multi-head models (single frame fallback)
if not ef_loaded:
    for ef_path in EF_RESNET_CANDIDATES:
        if not os.path.exists(ef_path):
            continue
        sd = _try_load_state_dict(ef_path)
        if sd is None:
            continue
        if _has_backbone_prefix(sd) and 'backbone.conv1.weight' in sd:
            try:
                mh = MultiHeadResNet.from_state_dict(sd, device=DEVICE)
                loaded_models["multihead"] = mh
                in_ch = sd['backbone.conv1.weight'].shape[1]
                heads = [k.replace('_head.weight','') for k in sd
                         if k.endswith('_head.weight') and sd[k].ndim == 2]
                model_info["multihead"] = {"in_channels": in_ch, "heads": heads}
                print(f"EF loaded (ResNet18 multi-head: {'+'.join(heads)}) from {Path(ef_path).name}")
                ef_loaded = True
                break
            except Exception as e:
                print(f"  {Path(ef_path).name} ResNet18 failed: {e}")

if not ef_loaded:
    print("No EF model could be loaded")

print(f"\nLoaded {len(loaded_models)} model(s): {list(loaded_models.keys())}")
print(f"Model info: {model_info}")

In [ ]:
# Cell [6] — Image Loading Helpers
# Supports images AND video clips from the ultrasound probe

def load_ultrasound_image(path, frame_index=None):
    """
    Load an ultrasound image from various formats, including MP4 video.

    For video files (.mp4, .avi, .mov):
      - Extracts a single frame (middle frame by default).
      - Set frame_index to pick a specific frame (0-based).

    Returns: (image_array, pixel_spacing)
        image_array: 2D numpy array (H, W), grayscale
        pixel_spacing: (dx, dy) in mm, or (1.0, 1.0) if unknown
    """
    path = str(path)
    ext = path.lower()

    if ext.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        import cv2
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise IOError(f"Cannot open video: {path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        target = frame_index if frame_index is not None else total_frames // 2
        target = max(0, min(target, total_frames - 1))

        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise IOError(f"Failed to read frame {target} from {path}")

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
        return img, (1.0, 1.0)

    elif ext.endswith(('.nii', '.nii.gz')):
        import nibabel as nib
        nii = nib.load(path)
        img = nii.get_fdata().astype(np.float32)
        if img.ndim == 3:
            img = img[:, :, img.shape[2] // 2]
        spacing = nii.header.get_zooms()[:2]
        return img, spacing

    elif ext.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        from PIL import Image
        img = np.array(Image.open(path).convert('L')).astype(np.float32)
        return img, (1.0, 1.0)

    elif ext.endswith('.dcm'):
        try:
            import pydicom
            ds = pydicom.dcmread(path)
            img = ds.pixel_array.astype(np.float32)
            spacing = getattr(ds, 'PixelSpacing', [1.0, 1.0])
            return img, (float(spacing[0]), float(spacing[1]))
        except ImportError:
            raise ImportError("Install pydicom: pip install pydicom")

    elif ext.endswith('.npy'):
        img = np.load(path).astype(np.float32)
        return img, (1.0, 1.0)

    else:
        raise ValueError(f"Unsupported format: {path}")


def get_video_info(path):
    """Print basic info about a video file."""
    import cv2
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"  Cannot open: {path}")
        return
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = frames / fps if fps > 0 else 0
    print(f"  {Path(path).name}: {w}x{h}, {frames} frames, {fps:.1f} fps, {duration:.1f}s")


print("Image/video loader ready")
print("  Supported: .mp4, .avi, .mov, .nii.gz, .nii, .png, .jpg, .dcm, .npy")
print("  Videos: extracts middle frame by default (set frame_index= to pick a specific frame)")

In [ ]:
# Cell [5b] — DIAGNOSTIC: What's going wrong?
# Run this cell to check every step of the pipeline.
# Share the FULL output so we can pinpoint the issue.

print("=" * 70)
print("  DIAGNOSTIC REPORT")
print("=" * 70)

# --- 1. Check loaded weight shapes ---
print("\n[1] SEGMENTATION WEIGHT SHAPES")
if "segmentation" in loaded_models:
    seg = loaded_models["segmentation"]
    for name, param in seg.named_parameters():
        print(f"    {name}: {list(param.shape)}")
    print(f"    Total params: {sum(p.numel() for p in seg.parameters()):,}")
else:
    print("    NOT LOADED")

print("\n[2] CLASSIFICATION CHECK")
for key in ("class_multihead", "classification"):
    if key in loaded_models:
        m = loaded_models[key]
        print(f"    Loaded as: {key}")
        if hasattr(m, 'head_names'):
            print(f"    Heads: {m.head_names}")
        print(f"    Params: {sum(p.numel() for p in m.parameters()):,}")
        break
else:
    print("    NOT LOADED")

print("\n[3] EF CHECK")
for key in ("ef_convnext", "multihead", "ef_standalone"):
    if key in loaded_models:
        print(f"    Loaded as: {key}")
        print(f"    Info: {model_info.get(key, {})}")
        break
else:
    print("    NOT LOADED")

# --- 2. Test with a real image ---
print("\n[4] LOADING A TEST FRAME")
import glob
test_files = sorted([f for f in glob.glob(os.path.join(IMAGE_DIR, "*")) if os.path.isfile(f)])
if not test_files:
    test_files = sorted([f for f in glob.glob(os.path.join(IMAGE_DIR, "**/*"), recursive=True) if os.path.isfile(f)])
if not test_files:
    print(f"    No files found in {IMAGE_DIR}!")
    print("    Set IMAGE_DIR in Cell [3] to your test_images folder.")
else:
    test_file = test_files[0]
    print(f"    Using: {test_file}")
    image, spacing = load_ultrasound_image(test_file)
    print(f"    Frame shape:    {image.shape}")
    print(f"    Pixel range:    min={image.min():.1f}, max={image.max():.1f}, mean={image.mean():.1f}")
    print(f"    Nonzero pixels: {(image > 0).sum()} / {image.size} ({100*(image>0).sum()/image.size:.1f}%)")

    # Show the frame
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(image, cmap='gray')
    ax.set_title(f"Loaded frame from {Path(test_file).name}")
    ax.axis('off')
    plt.show()

    # --- 3. Raw model outputs ---
    img = image.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_1ch = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
    img_1ch = F.resize(img_1ch, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR).to(DEVICE)

    print(f"\n[5] RAW SEGMENTATION OUTPUT")
    if "segmentation" in loaded_models:
        with torch.no_grad():
            seg_out = loaded_models["segmentation"](img_1ch)
        print(f"    Output shape: {list(seg_out.shape)}")
        print(f"    Raw values:   min={seg_out.min():.4f}, max={seg_out.max():.4f}, mean={seg_out.mean():.4f}")
        prob = torch.sigmoid(seg_out)
        print(f"    After sigmoid: min={prob.min():.4f}, max={prob.max():.4f}, mean={prob.mean():.4f}")
        mask = (prob > 0.5).sum().item()
        total = prob.numel()
        print(f"    Pixels > 0.5:  {mask} / {total} ({100*mask/total:.2f}%)")
        if seg_out.max() == seg_out.min():
            print("    *** ALL OUTPUT VALUES ARE IDENTICAL — model is not working! ***")
    else:
        print("    SKIPPED (not loaded)")

    print(f"\n[6] RAW CLASSIFICATION OUTPUT")
    if "classification" in loaded_models:
        in_ch = model_info.get("classification", {}).get("in_channels", 1)
        cls_input = img_1ch.expand(-1, in_ch, -1, -1) if in_ch > 1 else img_1ch
        with torch.no_grad():
            cls_out = loaded_models["classification"](cls_input)
        print(f"    Input shape:  {list(cls_input.shape)}")
        print(f"    Output shape: {list(cls_out.shape)}")
        print(f"    Raw logits:   {cls_out.squeeze().tolist()}")
        probs = torch.softmax(cls_out, dim=1).squeeze()
        print(f"    Softmax:      {[f'{p:.4f}' for p in probs.tolist()]}")
        print(f"    Predicted:    class {probs.argmax().item()}")
        if cls_out.max() - cls_out.min() < 0.001:
            print("    *** ALL LOGITS NEARLY IDENTICAL — model may not be working! ***")
    else:
        print("    SKIPPED (not loaded)")

    print(f"\n[7] RAW EF OUTPUT")
    if "ef" in loaded_models:
        in_ch = model_info.get("ef", {}).get("in_channels", 1)
        ef_input = img_1ch.expand(-1, in_ch, -1, -1) if in_ch > 1 else img_1ch
        with torch.no_grad():
            ef_out = loaded_models["ef"](ef_input)
        print(f"    Input shape:  {list(ef_input.shape)}")
        print(f"    Output shape: {list(ef_out.shape)}")
        print(f"    Raw value:    {ef_out.squeeze().tolist()}")
    else:
        print("    SKIPPED (not loaded)")

    # --- 4. Check if segmentation architecture might be wrong ---
    print(f"\n[8] ARCHITECTURE MATCH CHECK")
    if "segmentation" in loaded_models:
        sd = loaded_models["segmentation"].state_dict()
        up2_in = sd['up2.0.weight'].shape[1]
        mid_out = sd['mid.2.weight'].shape[0]
        down2_out = sd['down2.2.weight'].shape[0]
        expected_concat = mid_out + down2_out
        print(f"    up2.0 input channels:  {up2_in}")
        print(f"    mid output channels:   {mid_out}")
        print(f"    down2 output channels: {down2_out}")
        if up2_in == expected_concat:
            print(f"    Skip connections:      YES (concat {mid_out}+{down2_out}={expected_concat})")
        elif up2_in == mid_out:
            print(f"    Skip connections:      NO (up2 input = {up2_in} = mid output)")
            print(f"    *** ARCHITECTURE MISMATCH: SimpleUNet uses skip connections but model does not! ***")
        else:
            print(f"    *** UNEXPECTED: up2 input={up2_in}, expected {expected_concat} (skip) or {mid_out} (no skip) ***")

print("\n" + "=" * 70)
print("  Share this full output to diagnose the issue.")
print("=" * 70)

In [ ]:
# Cell [5c] — PREPROCESSING EXPERIMENT
# The segmentation model has NO BatchNorm, so it's very sensitive to input scale.
# The classification model (ResNet18 WITH BatchNorm) works fine — proving the
# loading code is correct. The segmentation model likely needs different input
# preprocessing than our default 0-to-1 normalization.
#
# This cell tests multiple preprocessing approaches to find which one
# actually activates the segmentation model.

import glob

test_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "**/*"), recursive=True))
test_files = [f for f in test_files if any(f.lower().endswith(e)
              for e in ('.avi','.mp4','.mov','.png','.jpg','.nii.gz','.nii','.npy'))]
if not test_files:
    print(f"No files in {IMAGE_DIR}. Update IMAGE_DIR in Cell [3].")
else:
    test_file = test_files[0]
    raw_image, spacing = load_ultrasound_image(test_file)
    print(f"Test image: {Path(test_file).name}")
    print(f"  Shape: {raw_image.shape}, range [{raw_image.min():.1f}, {raw_image.max():.1f}]\n")

    seg = loaded_models.get("segmentation")
    if seg is None:
        print("Segmentation model not loaded!")
    else:
        preprocessing_methods = {
            "A) 0-to-1 (current)":
                lambda img: (img - img.min()) / (img.max() - img.min() + 1e-8),
            "B) Raw 0-255":
                lambda img: img.copy(),
            "C) 0-255 / 255":
                lambda img: img / 255.0,
            "D) ImageNet-style (mean=0.449, std=0.226)":
                lambda img: ((img / 255.0) - 0.449) / 0.226,
            "E) Zero-mean unit-var":
                lambda img: (img - img.mean()) / (img.std() + 1e-8),
            "F) Clipped [1,254] then 0-to-1":
                lambda img: (np.clip(img, 1, 254) - 1) / 253.0,
        }

        results = {}
        print(f"{'Method':<42} {'Out min':>9} {'Out max':>9} {'Range':>9} {'Sig>0.5':>9}")
        print("-" * 82)

        for name, preprocess in preprocessing_methods.items():
            img = preprocess(raw_image.astype(np.float32))
            t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
            t = F.resize(t, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR).to(DEVICE)

            with torch.no_grad():
                out = seg(t)
            sig = torch.sigmoid(out)
            above = (sig > 0.5).sum().item()
            total = sig.numel()
            out_range = out.max().item() - out.min().item()

            results[name] = {
                'min': out.min().item(), 'max': out.max().item(),
                'range': out_range, 'above': above, 'total': total,
                'sig': sig.squeeze().cpu().numpy(),
                'out': out.squeeze().cpu().numpy(),
            }

            marker = " <-- HAS POSITIVE DETECTIONS" if above > 0 else ""
            print(f"{name:<42} {out.min().item():>9.4f} {out.max().item():>9.4f} "
                  f"{out_range:>9.4f} {above:>5}/{total}{marker}")

        # Show the best result (widest output range = most model activation)
        best = max(results, key=lambda k: results[k]['range'])
        print(f"\nBest activation: {best}  (output range = {results[best]['range']:.4f})")

        if results[best]['above'] > 0:
            print(f"  Detected {results[best]['above']} LV pixels!")

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            axes[0].imshow(raw_image, cmap='gray')
            axes[0].set_title("Input frame")
            axes[0].axis('off')

            axes[1].imshow(results[best]['out'], cmap='RdBu_r')
            axes[1].set_title(f"Raw output ({best})")
            axes[1].axis('off')
            plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046)

            mask = (results[best]['sig'] > 0.5).astype(float)
            axes[2].imshow(raw_image, cmap='gray')
            overlay = np.ma.masked_where(mask == 0, mask)
            axes[2].imshow(
                F.resize(torch.from_numpy(overlay).unsqueeze(0),
                         raw_image.shape[:2],
                         interpolation=F.InterpolationMode.NEAREST).squeeze().numpy(),
                cmap='Reds', alpha=0.5)
            axes[2].set_title("LV detection overlay")
            axes[2].axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print("\n  No preprocessing produced positive detections.")
            print("  This means either:")
            print("    1. The model was trained on very different data (not these ultrasound frames)")
            print("    2. The model needs additional preprocessing (cropping to ultrasound sector, etc.)")
            print("    3. The model weights may not be well-trained")
            print("\n  RECOMMENDED: Ask your teammate:")
            print("    - What exact preprocessing did you use before feeding images to this model?")
            print("    - Was this model trained on CAMUS .nii.gz data or on video frames?")
            print("    - What validation Dice/IoU score did this model achieve?")

In [ ]:
# Cell [5d] — EF MODEL DIAGNOSTIC
# Your teammate's code has EF working — let's figure out what input it expects.
# Tests: raw vs normalized, full frame vs LV-masked, and checks weight loading.

import glob

test_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "**/*"), recursive=True))
test_files = [f for f in test_files if any(f.lower().endswith(e)
              for e in ('.avi','.mp4','.mov','.png','.jpg','.nii.gz','.nii','.npy'))]

if "ef" not in loaded_models:
    print("EF model not loaded — cannot run diagnostic.")
elif "segmentation" not in loaded_models:
    print("Segmentation model not loaded — need it to create LV masks for testing.")
elif not test_files:
    print(f"No test files in {IMAGE_DIR}")
else:
    test_file = test_files[0]
    raw_image, spacing = load_ultrasound_image(test_file)
    print(f"Test: {Path(test_file).name}  shape={raw_image.shape}\n")

    # 1. Check weight loading integrity
    print("[1] EF MODEL WEIGHT LOADING CHECK")
    ef_sd_file = torch.load(EF_MODEL_PATH, map_location="cpu", weights_only=False)
    file_sd = _extract_state_dict(ef_sd_file)
    if file_sd is None and isinstance(ef_sd_file, dict):
        file_sd = ef_sd_file
    model_sd = loaded_models["ef"].state_dict()
    matched = 0
    mismatched = 0
    missing_in_model = 0
    for k in file_sd:
        if k in model_sd:
            if file_sd[k].shape == model_sd[k].shape:
                if torch.equal(file_sd[k].cpu(), model_sd[k].cpu()):
                    matched += 1
                else:
                    mismatched += 1
                    print(f"    KEY MISMATCH (values differ): {k}")
            else:
                mismatched += 1
                print(f"    SHAPE MISMATCH: {k}  file={list(file_sd[k].shape)} model={list(model_sd[k].shape)}")
        else:
            missing_in_model += 1
            print(f"    MISSING in model: {k}")
    print(f"    Matched: {matched}, Mismatched: {mismatched}, Missing: {missing_in_model}")
    if mismatched == 0 and missing_in_model == 0:
        print("    All weights loaded correctly!")
    else:
        print("    *** WEIGHT LOADING PROBLEM DETECTED ***")

    # 2. Get segmentation mask for this image
    pred_mask, _ = predict_segmentation(loaded_models["segmentation"], raw_image)

    # 3. Test EF with different input strategies
    print(f"\n[2] EF MODEL INPUT EXPERIMENTS")
    print(f"{'Input strategy':<45} {'Raw output':>12} {'As EF%':>8}")
    print("-" * 70)

    ef_m = loaded_models["ef"]
    in_ch = model_info.get("ef", {}).get("in_channels", 1)

    def run_ef(img_2d, label):
        t = torch.from_numpy(img_2d.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        if in_ch == 3:
            t = t.expand(-1, 3, -1, -1)
        t = F.resize(t, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR).to(DEVICE)
        with torch.no_grad():
            out = ef_m(t)
        val = out.squeeze().item()
        print(f"  {label:<43} {val:>12.6f} {val*100:>7.2f}%")
        return val

    # A) Current: 0-to-1 normalized full frame
    img_norm = (raw_image - raw_image.min()) / (raw_image.max() - raw_image.min() + 1e-8)
    run_ef(img_norm, "A) 0-to-1 full frame (current)")

    # B) Raw 0-255 full frame
    run_ef(raw_image, "B) Raw 0-255 full frame")

    # C) 0-to-1 normalized, LV region only (masked)
    masked = raw_image.copy()
    mask_resized = F.resize(
        torch.from_numpy(pred_mask.astype(np.float32)).unsqueeze(0),
        raw_image.shape[:2], interpolation=F.InterpolationMode.NEAREST
    ).squeeze().numpy()
    masked[mask_resized == 0] = 0
    masked_norm = (masked - masked.min()) / (masked.max() - masked.min() + 1e-8)
    run_ef(masked_norm, "C) 0-to-1 LV-masked image")

    # D) Raw 0-255, LV region only
    run_ef(masked, "D) Raw 0-255 LV-masked image")

    # E) Binary mask itself
    run_ef(mask_resized.astype(np.float32), "E) Binary LV mask (0/1)")

    # F) Binary mask * 255
    run_ef(mask_resized.astype(np.float32) * 255, "F) Binary LV mask (0/255)")

    # G) Zero-mean unit-var full frame
    img_z = (raw_image - raw_image.mean()) / (raw_image.std() + 1e-8)
    run_ef(img_z, "G) Zero-mean unit-var full frame")

    print("\n  Normal EF range: 50-70%. Look for which strategy gives values in that range.")
    print("  If NONE give reasonable values, the EF model may need a different architecture")
    print("  or a pipeline step we're missing. Ask your teammate to share the code that")
    print("  calls ef_trained.pt.")

In [ ]:
# Cell [7] — Segmentation Inference
# The segmentation model (SimpleUNet, no BatchNorm) expects RAW 0-255 pixel values.
# Discovered via Cell [5c] preprocessing experiment.

def predict_segmentation(seg_model, image_array):
    """
    Run segmentation on a single 2D ultrasound image.

    IMPORTANT: This model was trained on raw 0-255 pixel values (no normalization).

    Returns:
        pred_mask: 2D int array (H, W) with labels 0/1 (binary) or 0..N-1
        confidence: 2D float array (H, W) with prediction confidence
    """
    original_shape = image_array.shape[:2]

    img = image_array.astype(np.float32)
    # NO normalization — model expects raw 0-255 values

    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        output = seg_model(img_tensor)  # (1, C, H, W)

    n_classes = output.shape[1]

    if n_classes == 1:
        prob = torch.sigmoid(output)
        pred = (prob > 0.5).float()
        confidence = torch.where(pred == 1, prob, 1.0 - prob)
        pred = pred.squeeze(1)
        confidence = confidence.squeeze(1)
    else:
        probs = torch.softmax(output, dim=1)
        confidence, pred = torch.max(probs, dim=1)

    pred = F.resize(pred.unsqueeze(1).float(), original_shape,
                    interpolation=F.InterpolationMode.NEAREST)
    confidence = F.resize(confidence.unsqueeze(1).float(), original_shape,
                          interpolation=F.InterpolationMode.BILINEAR)

    return (
        pred.squeeze().cpu().numpy().astype(np.int64),
        confidence.squeeze().cpu().numpy(),
    )

seg_out_ch = model_info.get("segmentation", {}).get("out_channels", "?")
print(f"Segmentation inference ready ({seg_out_ch}-ch output, raw 0-255 input, no normalization)")

In [ ]:
# Cell [7b] — Classification, Quality, EF & LVOT Inference

def _prepare_resnet_input(image_array, model_key, normalize=True):
    """Normalize and shape an image for a ResNet model, handling 1-ch vs 3-ch input."""
    img = image_array.astype(np.float32)
    if normalize:
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    in_ch = model_info.get(model_key, {}).get("in_channels", 1)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    if in_ch == 3:
        img_tensor = img_tensor.expand(-1, 3, -1, -1)            # (1, 3, H, W)

    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    return img_tensor.to(DEVICE)


def predict_view_class(image_array):
    """
    Classify ultrasound view type.
    Returns (predicted_class_index, class_probabilities).
    Uses 'quality' head (3 classes) from class_multihead, or standalone classification.
    """
    if "class_multihead" in loaded_models:
        img_tensor = _prepare_resnet_input(image_array, "class_multihead")
        with torch.no_grad():
            outputs = loaded_models["class_multihead"](img_tensor)
        heads = model_info["class_multihead"]["heads"]
        cls_head = "quality" if "quality" in heads else list(heads.keys())[0]
        logits = outputs[cls_head]
        if logits.shape[-1] > 1:
            probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            return int(probs.argmax()), probs
        return 0, np.array([1.0])

    if "classification" in loaded_models:
        img_tensor = _prepare_resnet_input(image_array, "classification")
        with torch.no_grad():
            logits = loaded_models["classification"](img_tensor)
            probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        return int(probs.argmax()), probs

    return None, None


def _run_multihead(image_array):
    """
    Run the multi-head model (quality + EF + LVOT) on an image.
    Returns dict with 'quality', 'ef', 'lvot' values, or None if not loaded.
    """
    if "multihead" not in loaded_models:
        return None

    img_tensor = _prepare_resnet_input(image_array, "multihead")
    with torch.no_grad():
        outputs = loaded_models["multihead"](img_tensor)
    return {
        'quality': float(outputs['quality'].squeeze().cpu()),
        'ef': float(outputs['ef'].squeeze().cpu()),
        'lvot': float(outputs['lvot'].squeeze().cpu()),
    }


def predict_quality_score(image_array):
    """Predict image quality/grade score. Tries class_multihead grade head, then others."""
    if "class_multihead" in loaded_models:
        heads = model_info["class_multihead"]["heads"]
        if "grade" in heads:
            img_tensor = _prepare_resnet_input(image_array, "class_multihead")
            with torch.no_grad():
                out = loaded_models["class_multihead"](img_tensor)
            return float(out['grade'].squeeze().cpu())

    for key in ("quality_standalone", "quality_multihead"):
        if key in loaded_models:
            img_tensor = _prepare_resnet_input(image_array, key)
            with torch.no_grad():
                out = loaded_models[key](img_tensor)
            if isinstance(out, dict):
                return float(out.get('quality', out.get('grade', list(out.values())[0])).squeeze().cpu())
            return float(out.squeeze().cpu())

    result = _run_multihead(image_array)
    if result and 'quality' in result:
        return result['quality']
    return None


def predict_ef_from_video(video_path):
    """
    Predict EF using the ConvNeXt video model (16 frames).
    This is the most accurate EF model — same as teammate's code.
    Returns float EF value, or None if model not loaded.
    """
    if "ef_convnext" not in loaded_models:
        return None
    video_tensor = read_video_for_ef(video_path, N_FRAMES)   # (3, T, 224, 224)
    video_tensor = video_tensor.unsqueeze(0).to(DEVICE)       # (1, 3, T, 224, 224)
    with torch.no_grad():
        ef_val = loaded_models["ef_convnext"](video_tensor)
    return float(ef_val.squeeze().cpu())


def predict_ef_direct(image_array, video_path=None):
    """
    Predict EF. Tries ConvNeXt video model first (if video_path given),
    then multi-head ResNet18, then standalone.
    """
    if video_path and "ef_convnext" in loaded_models:
        return predict_ef_from_video(video_path)

    result = _run_multihead(image_array)
    if result:
        return result['ef']

    if "ef_standalone" in loaded_models:
        img_tensor = _prepare_resnet_input(image_array, "ef_standalone")
        with torch.no_grad():
            out = loaded_models["ef_standalone"](img_tensor)
        return float(out.squeeze().cpu())
    return None


def predict_lvot(image_array):
    """Predict LVOT measurement from multi-head model."""
    result = _run_multihead(image_array)
    if result:
        return result['lvot']
    return None


available = []
if "classification" in loaded_models:
    n = model_info["classification"]["n_classes"]
    available.append(f"View classification ({n} classes)")
if "multihead" in loaded_models:
    heads = model_info["multihead"]["heads"]
    available.append(f"Multi-head: {'+'.join(heads)}")
if "ef_standalone" in loaded_models:
    available.append("EF (standalone)")
for qk in ("quality_standalone", "quality_multihead"):
    if qk in loaded_models:
        available.append(f"Quality ({qk.split('_')[1]})")
        break
print(f"Inference functions ready: {available or ['Only segmentation']}")

In [ ]:
# Cell [8] — Cardiac Metrics Calculation
# Works for both binary masks (label 1 = LV) and multi-class masks (1=LV, 2=Myo, 3=LA)

def calculate_areas(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Calculate areas of each segmented structure in mm2."""
    dx, dy = pixel_spacing_mm
    pixel_area = dx * dy

    areas = {'lv_area_mm2': float((pred_mask == 1).sum() * pixel_area)}
    if pred_mask.max() > 1:
        areas['myocardium_area_mm2'] = float((pred_mask == 2).sum() * pixel_area)
        areas['la_area_mm2'] = float((pred_mask == 3).sum() * pixel_area)
    return areas


def calculate_lv_length_mm(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Estimate LV long-axis length from segmentation mask."""
    dx, dy = pixel_spacing_mm
    ys, xs = np.where(pred_mask == 1)

    if len(xs) == 0:
        return None

    length_pixels = np.sqrt((xs.max() - xs.min())**2 + (ys.max() - ys.min())**2)
    return float(length_pixels * np.mean([dx, dy]))


def calculate_volume_biplane(area_2ch_mm2, area_4ch_mm2, length_mm):
    """
    Biplane area-length method (modified Simpson's):
    V = (8 / 3π) × (A_2CH × A_4CH) / L
    Returns volume in mL.
    """
    if None in [area_2ch_mm2, area_4ch_mm2, length_mm] or length_mm == 0:
        return None
    
    volume_mm3 = (8 / (3 * np.pi)) * (area_2ch_mm2 * area_4ch_mm2) / length_mm
    return volume_mm3 / 1000  # mm³ → mL


def calculate_ef(edv_ml, esv_ml):
    """Ejection Fraction (%)."""
    if edv_ml is None or esv_ml is None or edv_ml == 0:
        return None
    return ((edv_ml - esv_ml) / edv_ml) * 100


def calculate_cardiac_output(sv_ml, heart_rate_bpm):
    """Cardiac Output in L/min."""
    return (sv_ml * heart_rate_bpm) / 1000


def calculate_vti(sv_ml, lvot_diameter_cm=2.0):
    """
    Velocity Time Integral (cm).
    VTI = SV / LVOT_area
    """
    lvot_area = np.pi * (lvot_diameter_cm / 2)**2
    return sv_ml / lvot_area  # mL = cm³, so result is in cm


print("✅ Cardiac metrics functions ready")

In [ ]:
# Cell [9] — Visualization
# Handles both binary (1-class) and multi-class segmentation

def visualize_result(image, pred_mask, confidence, title=""):
    """Display original image, segmentation, and overlay."""
    is_binary = pred_mask.max() <= 1

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    if is_binary:
        axes[1].imshow(pred_mask, cmap='Reds', vmin=0, vmax=1)
        axes[1].set_title('Segmentation (LV)')
    else:
        axes[1].imshow(pred_mask, cmap='tab10', vmin=0, vmax=pred_mask.max())
        axes[1].set_title('Segmentation')
    axes[1].axis('off')

    axes[2].imshow(image, cmap='gray')
    if is_binary:
        overlay = np.ma.masked_where(pred_mask == 0, pred_mask)
        axes[2].imshow(overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
        axes[2].set_title('Overlay (Red = LV)')
    else:
        for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
            overlay = np.ma.masked_where(pred_mask != label, pred_mask)
            axes[2].imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)
        axes[2].set_title('Overlay (R=LV, B=Myo, G=LA)')
    axes[2].axis('off')

    im = axes[3].imshow(confidence, cmap='RdYlGn', vmin=0.5, vmax=1.0)
    axes[3].set_title(f'Confidence (avg: {confidence.mean():.2f})')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    if title:
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    plt.tight_layout()
    plt.show()

print("Visualization functions ready")

In [ ]:
# Cell [10] — Run on a Single Clip (all models)
# Your 15 probe clips: clip1.mp4 .. clip15.mp4  (plus clip3.png)
# Change the filename below to test any single clip.

test_clip = os.path.join(IMAGE_DIR, "clip1.mp4")  # <-- change clip number here

if os.path.exists(test_clip):
    # Show video info if it's an mp4
    if test_clip.lower().endswith(('.mp4', '.avi', '.mov')):
        get_video_info(test_clip)

    image, spacing = load_ultrasound_image(test_clip)
    print(f"\nLoaded: {test_clip}")
    print(f"  Frame shape: {image.shape}, Pixel spacing: {spacing} mm")

    # 1 — View classification
    view_idx, view_probs = predict_view_class(image)
    if view_idx is not None:
        print(f"\n[Classification] Predicted view index: {view_idx}  (confidence: {view_probs[view_idx]:.2f})")
    else:
        print("\n[Classification] Model not loaded — skipped")

    # 2 — Multi-head predictions (Quality, EF, LVOT)
    multihead_result = _run_multihead(image)
    if multihead_result is not None:
        print(f"\n[Multi-head model]")
        print(f"  Quality Score: {multihead_result['quality']:.3f}")
        print(f"  EF:            {multihead_result['ef']:.2f}%")
        print(f"  LVOT:          {multihead_result['lvot']:.3f}")
    else:
        quality = predict_quality_score(image)
        if quality is not None:
            print(f"[Quality]        Score: {quality:.3f}")
        else:
            print("[Quality]        Model not loaded — skipped")

    # 3 — Segmentation
    if "segmentation" in loaded_models:
        pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
        areas = calculate_areas(pred_mask, spacing)
        lv_length = calculate_lv_length_mm(pred_mask, spacing)
        print(f"\n[Segmentation]")
        print(f"  LV Cavity Area:    {areas['lv_area_mm2']:.1f} mm2")
        if 'myocardium_area_mm2' in areas:
            print(f"  Myocardium Area:   {areas['myocardium_area_mm2']:.1f} mm2")
            print(f"  Left Atrium Area:  {areas['la_area_mm2']:.1f} mm2")
        if lv_length:
            print(f"  LV Length:         {lv_length:.1f} mm")
        print(f"  Avg Confidence:    {confidence.mean():.2f}")
        visualize_result(image, pred_mask, confidence, title=Path(test_clip).name)
    else:
        print("\n[Segmentation]   Model not loaded — skipped")

    # 4 — Video-based EF (ConvNeXt, 16 frames — most accurate)
    ef_video = predict_ef_from_video(test_clip) if "ef_convnext" in loaded_models else None
    if ef_video is not None:
        print(f"\n[EF Video]       Predicted EF: {ef_video:.1f}%  (ConvNeXt, {N_FRAMES} frames)")
    elif multihead_result is None:
        ef_direct = predict_ef_direct(image)
        if ef_direct is not None:
            print(f"[EF ResNet]      Predicted EF: {ef_direct:.1f}%  (single-frame fallback)")
        else:
            print("[EF]             No EF model loaded — skipped")
else:
    print(f"File not found: {test_clip}")
    print(f"Make sure IMAGE_DIR in Cell [3] points to your test_images/ folder")

In [ ]:
# Cell [11] — Full Pipeline: Process a Set of 15 Probe Images (all models)

def process_all_images(image_dir):
    """
    Run every loaded model on all ultrasound images in a directory.
    Returns a list of per-image result dicts.
    """
    supported_ext = ('.mp4', '.avi', '.mov', '.nii.gz', '.nii', '.png', '.jpg', '.jpeg', '.dcm', '.npy')

    image_files = []
    for f in sorted(os.listdir(image_dir)):
        if any(f.lower().endswith(ext) for ext in supported_ext):
            image_files.append(os.path.join(image_dir, f))

    if not image_files:
        print(f"No supported images found in {image_dir}")
        return []

    print(f"Found {len(image_files)} images in {image_dir}\n")

    results = []
    for i, img_path in enumerate(image_files, 1):
        fname = Path(img_path).name
        print(f"[{i}/{len(image_files)}] {fname}")

        try:
            image, spacing = load_ultrasound_image(img_path)
            entry = {
                'file': fname,
                'image': image,
                'spacing': spacing,
            }

            # Classification
            view_idx, view_probs = predict_view_class(image)
            entry['view_class'] = view_idx
            entry['view_probs'] = view_probs

            # EF prediction (ConvNeXt video model preferred, then multi-head fallback)
            entry['ef_direct'] = predict_ef_direct(image, video_path=img_path)

            # Quality + LVOT from multi-head
            mh = _run_multihead(image)
            if mh:
                entry['quality_score'] = mh.get('quality')
                entry['lvot'] = mh.get('lvot')
            else:
                entry['quality_score'] = predict_quality_score(image)
                entry['lvot'] = None

            # Segmentation
            if "segmentation" in loaded_models:
                pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
                areas = calculate_areas(pred_mask, spacing)
                entry.update({
                    'pred_mask': pred_mask,
                    'confidence': confidence,
                    'areas': areas,
                    'lv_length_mm': calculate_lv_length_mm(pred_mask, spacing),
                    'avg_confidence': float(confidence.mean()),
                    'has_lv': bool((pred_mask == 1).any()),
                    'has_myocardium': bool((pred_mask == 2).any()),
                    'has_la': bool((pred_mask == 3).any()),
                })
                seg_info = f"LV={areas['lv_area_mm2']:.0f}mm2"
            else:
                seg_info = "seg N/A"

            qual_str = f"Q={entry['quality_score']:.2f}" if entry['quality_score'] is not None else "Q=N/A"
            view_str = f"View={view_idx}" if view_idx is not None else "View=N/A"
            ef_str = f"EF={entry['ef_direct']:.1f}%" if entry['ef_direct'] is not None else "EF=N/A"
            lvot_str = f"LVOT={entry['lvot']:.2f}" if entry.get('lvot') is not None else ""
            print(f"   {view_str} | {qual_str} | {seg_info} | {ef_str} {lvot_str}")

            results.append(entry)

        except Exception as e:
            print(f"   Error: {e}")
            results.append({'file': fname, 'error': str(e)})

    return results

if os.path.isdir(IMAGE_DIR):
    all_results = process_all_images(IMAGE_DIR)
else:
    print(f"Directory not found: {IMAGE_DIR}")
    print(f"Create the folder and add your ultrasound images, or update IMAGE_DIR in Cell [3]")
    all_results = []

In [ ]:
# Cell [12] — Full Cardiac Report (4-Chamber View)
# All clips are A4C (4-chamber). EF comes from the ConvNeXt video model.
# Volumes, CO, and VTI are derived from EF + clinical parameters below.

# =====================================================
# CLINICAL PARAMETERS — adjust for your patient
# =====================================================
HEART_RATE_BPM = 75        # beats per minute
LVOT_DIAMETER_CM = 2.0     # left ventricular outflow tract diameter (cm)
ASSUMED_EDV_ML = 120.0     # assumed end-diastolic volume (mL) — normal adult ~120

VIEW_LABELS = {0: "A2C", 1: "PLAX", 2: "A4C"}

def compute_full_cardiac_report(results, hr=HEART_RATE_BPM,
                                 lvot_d=LVOT_DIAMETER_CM, edv=ASSUMED_EDV_ML):
    valid = [r for r in results if 'error' not in r]
    if not valid:
        print("No valid results to analyze")
        return None

    print("\n" + "=" * 80)
    print("  CARDIAC REPORT  —  4-Chamber (A4C) View")
    print("=" * 80)

    # --- Per-clip table ---
    header = (f"{'#':<3} {'File':<20} {'View':<6} {'Qual':>6} "
              f"{'LV px':>8} {'Conf':>6} {'EF%':>7}")
    print(f"\n{header}")
    print("-" * len(header))

    ef_values = []
    for i, r in enumerate(valid):
        view_idx = r.get('view_class')
        view_str = VIEW_LABELS.get(view_idx, str(view_idx)) if view_idx is not None else "--"
        qual = r.get('quality_score')
        qual_str = f"{qual:>6.2f}" if qual is not None else f"{'--':>6}"
        lv = r.get('areas', {}).get('lv_area_mm2')
        lv_str = f"{lv:>8.0f}" if lv is not None else f"{'--':>8}"
        conf = r.get('avg_confidence')
        conf_str = f"{conf:>6.2f}" if conf is not None else f"{'--':>6}"
        ef = r.get('ef_direct')
        ef_str = f"{ef:>7.1f}" if ef is not None else f"{'--':>7}"
        if ef is not None:
            ef_values.append(ef)
        print(f"{i:<3} {r['file']:<20} {view_str:<6} {qual_str} {lv_str} {conf_str} {ef_str}")

    # --- Computed cardiac metrics ---
    if ef_values:
        avg_ef = np.mean(ef_values)
        ef_frac = avg_ef / 100.0

        esv = edv * (1 - ef_frac)
        sv = edv - esv
        co = (sv * hr) / 1000
        lvot_area = np.pi * (lvot_d / 2) ** 2
        vti = sv / lvot_area

        print("\n" + "=" * 80)
        print("  CARDIAC METRICS")
        print("=" * 80)
        print(f"\n  Ejection Fraction (EF):     {avg_ef:.1f}%   (avg of {len(ef_values)} clips)")
        print(f"  End-Diastolic Volume (EDV):  {edv:.1f} mL   (assumed)")
        print(f"  End-Systolic Volume (ESV):   {esv:.1f} mL   (= EDV x (1 - EF))")
        print(f"  Stroke Volume (SV):          {sv:.1f} mL   (= EDV - ESV)")
        print(f"  Heart Rate (HR):             {hr} bpm  (assumed)")
        print(f"  Cardiac Output (CO):         {co:.2f} L/min (= SV x HR / 1000)")
        print(f"  LVOT Diameter:               {lvot_d:.1f} cm  (assumed)")
        print(f"  LVOT Area:                   {lvot_area:.2f} cm2")
        print(f"  VTI:                         {vti:.1f} cm   (= SV / LVOT area)")

        print(f"\n  {'Metric':<30} {'Value':>10} {'Normal Range':>20}")
        print(f"  {'-'*62}")
        print(f"  {'Ejection Fraction':<30} {avg_ef:>9.1f}% {'55 - 70%':>20}")
        print(f"  {'Stroke Volume':<30} {sv:>9.1f} mL {'60 - 100 mL':>20}")
        print(f"  {'Cardiac Output':<30} {co:>9.2f} L/min {'4.0 - 8.0 L/min':>20}")
        print(f"  {'VTI':<30} {vti:>9.1f} cm {'18 - 22 cm':>20}")

        if avg_ef >= 55:
            print(f"\n  Assessment: EF is NORMAL (>= 55%)")
        elif avg_ef >= 40:
            print(f"\n  Assessment: EF is MILDLY REDUCED (40-54%)")
        elif avg_ef >= 30:
            print(f"\n  Assessment: EF is MODERATELY REDUCED (30-39%)")
        else:
            print(f"\n  Assessment: EF is SEVERELY REDUCED (< 30%)")

        print(f"\n  Note: EDV is assumed at {edv} mL. For accurate SV/CO/VTI,")
        print(f"  measure actual EDV from calibrated images or set ASSUMED_EDV_ML above.")
    else:
        print("\n  No EF predictions available.")

    return valid

if all_results:
    report = compute_full_cardiac_report(all_results)

In [ ]:
# Cell [13] — Visualize All Results in a Grid

def visualize_all_results(results, cols=5):
    """Display all segmentation results in a grid."""
    valid = [r for r in results if 'error' not in r]
    
    if not valid:
        print("No valid results to display")
        return
    
    n = len(valid)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if cols > 1 else np.array([[axes]])
    
    for i, r in enumerate(valid):
        row, col = i // cols, i % cols
        ax = axes[row, col]
        
        ax.imshow(r['image'], cmap='gray')
        if 'pred_mask' in r:
            mask = r['pred_mask']
            if mask.max() <= 1:
                overlay = np.ma.masked_where(mask == 0, mask)
                ax.imshow(overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
            else:
                for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
                    overlay = np.ma.masked_where(mask != label, mask)
                    ax.imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)

        conf_str = f"{r['avg_confidence']:.2f}" if 'avg_confidence' in r else "N/A"
        ax.set_title(f"{r['file']}\nConf: {conf_str}", fontsize=9)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')
    
    plt.suptitle("Segmentation Results — All Images", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if all_results:
    visualize_all_results(all_results)